# Outlier Detection and Handling

Outliers are extreme values that differ significantly from other observations in a dataset. They can distort analysis and model performance, so detection and appropriate handling is important.

## Why Detect Outliers?
- Improve model accuracy
- Identify data quality issues
- Discover interesting patterns
- Prevent skewed statistics

## Detection Methods:
1. **Statistical Methods** (Z-Score, IQR)
2. **Distance-Based Methods** (Mahalanobis)
3. **Density-Based Methods** (Local Outlier Factor)
4. **Isolation Forest** (Tree-based)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from scipy import stats

# Load dataset
df = pd.read_csv('/Users/thameem/Desktop/thameem/Machine Learning/Data/Algerian_forest_fires_dataset_UPDATE.csv')
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData summary:")
print(df.describe())

## 1. Z-Score Method

Identifies outliers as values more than 3 standard deviations from the mean.

In [ ]:
# Z-Score method
numeric_cols = df.select_dtypes(include=[np.number]).columns
z_scores = np.abs(stats.zscore(df[numeric_cols]))

# Identify outliers (threshold = 3)
threshold = 3
outliers_zscore = (z_scores > threshold).any(axis=1)

print(f"Z-Score Method:")
print(f"Number of outliers detected: {outliers_zscore.sum()}")
print(f"Percentage of outliers: {(outliers_zscore.sum() / len(df)) * 100:.2f}%")

# Show some outlier examples
print("\nSample outliers:")
print(df[outliers_zscore].head())

## 2. Interquartile Range (IQR) Method

Uses quartiles to identify outliers: outliers are values beyond Q1 - 1.5*IQR or Q3 + 1.5*IQR

In [ ]:
# IQR Method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Apply IQR method to first numeric column
first_numeric_col = numeric_cols[0]
outliers_iqr = detect_outliers_iqr(df, first_numeric_col)

print(f"\nIQR Method for column '{first_numeric_col}':")
print(f"Q1: {df[first_numeric_col].quantile(0.25):.2f}")
print(f"Q3: {df[first_numeric_col].quantile(0.75):.2f}")
print(f"IQR: {df[first_numeric_col].quantile(0.75) - df[first_numeric_col].quantile(0.25):.2f}")
print(f"Number of outliers: {outliers_iqr.sum()}")

## 3. Isolation Forest

An ensemble method that isolates outliers by randomly selecting features and split values.

In [ ]:
# Isolation Forest
iso_forest = IsolationForest(contamination=0.1, random_state=42)
outliers_if = iso_forest.fit_predict(df[numeric_cols])

# -1 indicates outlier, 1 indicates inlier
outliers_if_bool = outliers_if == -1

print(f"\nIsolation Forest:")
print(f"Number of outliers detected: {outliers_if_bool.sum()}")
print(f"Percentage of outliers: {(outliers_if_bool.sum() / len(df)) * 100:.2f}%")

## 4. Handling Outliers

Options for dealing with detected outliers.

In [ ]:
# Option 1: Remove outliers
df_removed = df[~outliers_zscore]
print(f"After removing outliers (Z-Score):")
print(f"Original shape: {df.shape}")
print(f"New shape: {df_removed.shape}")
print(f"Rows removed: {df.shape[0] - df_removed.shape[0]}")

# Option 2: Cap outliers (Winsorization)
df_capped = df.copy()
for col in numeric_cols:
    lower_limit = df[col].quantile(0.01)
    upper_limit = df[col].quantile(0.99)
    df_capped[col] = df_capped[col].clip(lower=lower_limit, upper=upper_limit)

print(f"\nAfter capping outliers:")
print(f"Shape remains: {df_capped.shape}")
print("Values capped to 1st and 99th percentiles")